In [35]:
from prosperity3bt.data import LIMITS, BacktestData, read_day_data
from prosperity3bt.file_reader import FileReader, FileSystemReader, PackageResourcesReader
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import os
# import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# `PICNIC_BASKET1` contains three products: 

# 1. Six (6) `CROISSANTS`
# 2. Three (3) `JAMS`
# 3. One (1) `DJEMBE`

# `PICNIC_BASKET2` contains just two products: 

# 1. Four (4) `CROISSANTS`
# 2. Two (2) `JAMS`
to_calc_products = [
    "PICNIC_BASKET1",
    "PICNIC_BASKET2",
    "CROISSANTS",
    "DJEMBES",
    "JAMS"
]

In [3]:
from pathlib import Path


# Convert the relative path to an absolute path
path = "../../prosperity3bt/resources/round2/prices_round_2_day_-1.csv"
print(str(path))
# Check if the file path exists
if os.path.exists(path):  # Replace with the actual file path
    print("The file path exists.")
else:
    print("The file path does not exist.")

../../prosperity3bt/resources/round2/prices_round_2_day_-1.csv
The file path exists.


In [4]:
file_path = path  # Replace with the actual file path
data = pd.read_csv(file_path, sep=';')

In [5]:
data

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-1,0,CROISSANTS,4304,112,NaN,NaN,NaN,NaN,4305,112,NaN,NaN,NaN,NaN,4304.5,0.0
1,-1,0,JAMS,6670,66,6669.0,137.0,NaN,NaN,6671,66,6672.0,137.0,NaN,NaN,6670.5,0.0
2,-1,0,SQUID_INK,2005,1,2002.0,31.0,NaN,NaN,2006,31,NaN,NaN,NaN,NaN,2005.5,0.0
3,-1,0,PICNIC_BASKET1,59284,20,59283.0,18.0,NaN,NaN,59294,2,59295.0,36.0,NaN,NaN,59289.0,0.0
4,-1,0,PICNIC_BASKET2,30606,20,30605.0,18.0,NaN,NaN,30612,20,30613.0,18.0,NaN,NaN,30609.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,-1,999900,DJEMBES,13493,76,NaN,NaN,NaN,NaN,13494,76,NaN,NaN,NaN,NaN,13493.5,0.0
79996,-1,999900,KELP,2028,31,NaN,NaN,NaN,NaN,2030,6,2031.0,31.0,NaN,NaN,2029.0,0.0
79997,-1,999900,RAINFOREST_RESIN,9996,2,9995.0,29.0,NaN,NaN,10004,2,10005.0,29.0,NaN,NaN,10000.0,0.0
79998,-1,999900,PICNIC_BASKET1,59222,2,59221.0,13.0,59220.0,27.0,59231,2,59232.0,13.0,59233.0,27.0,59226.5,0.0


In [6]:
data['mean_price'] = (
    data[['bid_price_1', 'bid_price_2', 'bid_price_3']].min(axis=1, skipna=True) +
    data[['ask_price_1', 'ask_price_2',
                   'ask_price_3']].max(axis=1, skipna=True)
) / 2

In [14]:
product_mean_values = data[['timestamp', 'product', 'mean_price']]
price_dict= {}
for prod in to_calc_products:
    price_dict[prod] = product_mean_values[product_mean_values['product'] == prod]


In [25]:
price_dict["CROISSANTS"]

,timestamp,product,mean_price
0,0,CROISSANTS,4304.5
12,100,CROISSANTS,4304.5
19,200,CROISSANTS,4304.5
30,300,CROISSANTS,4304.5
33,400,CROISSANTS,4305.5
...,...,...,...
79960,999500,CROISSANTS,4322.0
79974,999600,CROISSANTS,4321.5
79979,999700,CROISSANTS,4321.5
79986,999800,CROISSANTS,4322.0


In [75]:
# Specify the products for which you want the summation
# to check price correlation using graph
# `PICNIC_BASKET1` contains three products:

# 1. Six (6) `CROISSANTS`
# 2. Three (3) `JAMS`
# 3. One (1) `DJEMBE`
basketsyn= ["CROISSANTS", "JAMS", "DJEMBES"]
prod1 = "PICNIC_BASKET1"
prod2 = "PICNIC_BASKET2"

# Filter the DataFrame for the specified products
filtered_products = product_mean_values[product_mean_values['product'].isin(basketsyn)]

# Summing up the mean prices of the specified products for each timestamp
summed_mean_prices = filtered_products.groupby('timestamp').apply(
    lambda x: 6 * x[x['product'] == "CROISSANTS"]['mean_price'].sum() +
              3 * x[x['product'] == "JAMS"]['mean_price'].sum()+
                1 * x[x['product'] == "DJEMBES"]['mean_price'].sum()
).reset_index(name='mean_price')

# Creating a new DataFrame
summed_mean_prices_df = pd.DataFrame(summed_mean_prices)

# Display the new DataFrame
print(summed_mean_prices_df)


def make_cor():
    # to make synthetic basket1
    fig = px.line(summed_mean_prices, x='timestamp', y='mean_price',
                  title='Price of PICNIC_BASKET1 over Time')
    fig.add_scatter(x=price_dict["PICNIC_BASKET1"]['timestamp'],
                    y=price_dict["PICNIC_BASKET1"]['mean_price'],
                    mode='lines',
                    name='Actual PICNIC_BASKET1')
    fig.show()

make_cor()

      timestamp  mean_price
0             0     59289.0
1           100     59288.5
2           200     59290.0
3           300     59293.0
4           400     59301.5
...         ...         ...
9995     999500     59321.0
9996     999600     59319.0
9997     999700     59318.5
9998     999800     59321.0
9999     999900     59318.5

[10000 rows x 2 columns]


C:\Users\MY PC\AppData\Local\Temp\ipykernel_22156\35494418.py:16: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [80]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Example function implementing pairs trading


def pairs_trading_basket(summed_mean_prices_df, basket1_df, window=1000, z_threshold=2.0):
    """
    Summarize:
      1. Merges synthetic (summed_mean_prices_df) & actual (basket1_df)
      2. Computes rolling z-score of the spread
      3. Generates simple trading signals
      4. Plots the results
    """
    # Merge
    df_pairs = pd.merge(
        summed_mean_prices_df,
        basket1_df,
        on='timestamp',
        how='inner',
        suffixes=('_synthetic', '_actual')
    )

    # Spread
    df_pairs['spread'] = df_pairs['mean_price_actual'] - \
        df_pairs['mean_price_synthetic']

    # Rolling stats
    df_pairs['spread_mean'] = df_pairs['spread'].rolling(window).mean()
    df_pairs['spread_std'] = df_pairs['spread'].rolling(window).std()

    # Z-score
    df_pairs['zscore'] = (df_pairs['spread'] -
                          df_pairs['spread_mean']) / df_pairs['spread_std']

    # Threshold-based signals
    df_pairs['signal'] = 0
    df_pairs.loc[df_pairs['zscore'] > z_threshold, 'signal'] = -1
    df_pairs.loc[df_pairs['zscore'] < -z_threshold, 'signal'] = 1

    # --- Plotting ---
    fig = go.Figure()

    # Synthetic Basket
    fig.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs['mean_price_synthetic'],
        mode='lines',
        name='Synthetic Basket'
    ))

    # Actual Basket
    fig.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs['mean_price_actual'],
        mode='lines',
        name='Actual PICNIC_BASKET1'
    ))

    # Identify buy/sell signals
    buy_signals = df_pairs[df_pairs['signal'] == 1]
    sell_signals = df_pairs[df_pairs['signal'] == -1]

    fig.add_trace(go.Scatter(
        x=buy_signals['timestamp'],
        y=buy_signals['mean_price_actual'],
        mode='markers',
        marker_symbol='triangle-up',
        marker_color='green',
        marker_size=12,
        name='Buy (Long Actual)'
    ))

    fig.add_trace(go.Scatter(
        x=sell_signals['timestamp'],
        y=sell_signals['mean_price_actual'],
        mode='markers',
        marker_symbol='triangle-down',
        marker_color='red',
        marker_size=12,
        name='Sell (Short Actual)'
    ))

    fig.update_layout(
        title='Pairs Trading Signals: Actual vs Synthetic',
        xaxis_title='Timestamp',
        yaxis_title='Price'
    )

    fig.show()

    return df_pairs


# Usage:
df_signals = pairs_trading_basket(
    summed_mean_prices_df,
    price_dict["PICNIC_BASKET1"],
    window=500,
    z_threshold=2
)

In [78]:
window= 1000
z_threshold = 2
df_pairs = pd.merge(
        summed_mean_prices_df,
        price_dict["PICNIC_BASKET1"],
        on='timestamp',
        how='inner',
        suffixes=('_synthetic', '_actual')
    )

# Spread
df_pairs['spread'] = df_pairs['mean_price_actual'] - \
    df_pairs['mean_price_synthetic']

# Rolling stats
df_pairs['spread_mean'] = df_pairs['spread'].rolling(window).mean()
df_pairs['spread_std'] = df_pairs['spread'].rolling(window).std()

# Z-score
df_pairs['zscore'] = (df_pairs['spread'] -
                        df_pairs['spread_mean']) / df_pairs['spread_std']

# Threshold-based signals
df_pairs['signal'] = 0
df_pairs.loc[df_pairs['zscore'] > z_threshold, 'signal'] = -1
df_pairs.loc[df_pairs['zscore'] < -z_threshold, 'signal'] = 1

# --- Plotting ---
# fig = go.Figure()
# Create a figure with 2 rows and shared x-axis
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=("Prices: Actual vs Synthetic", "Spread Z-score"))

# Row 1: Plot prices
fig2.add_trace(
    go.Scatter(x=df_pairs['timestamp'],
               y=df_pairs['mean_price_actual'],
               mode='lines',
               name='Actual PICNIC_BASKET1'),
    row=1, col=1
)
fig2.add_trace(
    go.Scatter(x=df_pairs['timestamp'],
               y=df_pairs['mean_price_synthetic'],
               mode='lines',
               name='Synthetic Basket'),
    row=1, col=1
)

# Row 2: Plot z-score
fig2.add_trace(
    go.Scatter(x=df_pairs['timestamp'],
               y=df_pairs['zscore'],
               mode='lines',
               name='Spread Z-score'),
    row=2, col=1
)

# Add threshold lines for reference
fig2.add_hrect(y0=z_threshold, y1=-z_threshold,
               fillcolor="green", opacity=0.2,
               line_width=0, row=2, col=1)

fig2.update_layout(
    height=700,
    title='Pairs Trading with Spread Z-score'
)
fig2.show()

In [56]:
# Specify the products for which you want the summation
# to check price correlation using graph

# `PICNIC_BASKET2` contains just two products:

# 1. Four (4) `CROISSANTS`
# 2. Two (2) `JAMS`
basketsyn = ["CROISSANTS", "JAMS"]

# Filter the DataFrame for the specified products
filtered_products = product_mean_values[product_mean_values['product'].isin(
    basketsyn)]

# Summing up the mean prices of the specified products for each timestamp
summed_mean_prices = filtered_products.groupby('timestamp').apply(
    lambda x: 4 * x[x['product'] == "CROISSANTS"]['mean_price'].sum() +
    2 * x[x['product'] == "JAMS"]['mean_price'].sum()
).reset_index(name='mean_price')

# Creating a new DataFrame
summed_mean_prices_df = pd.DataFrame(summed_mean_prices)

# Display the new DataFrame
print(summed_mean_prices_df)
print(summed_mean_prices_df)


def make_cor():
    # to make synthetic basket1
    fig = px.line(summed_mean_prices, x='timestamp', y='mean_price',
                  title='Price of PICNIC_BASKET2 over Time')
    fig.add_scatter(x=price_dict["PICNIC_BASKET2"]['timestamp'],
                    y=price_dict["PICNIC_BASKET2"]['mean_price'],
                    mode='lines',
                    name='Actual PICNIC_BASKET2')
    fig.show()


make_cor()

      timestamp  mean_price
0             0     30559.0
1           100     30559.0
2           200     30560.0
3           300     30562.0
4           400     30566.0
...         ...         ...
9995     999500     30552.0
9996     999600     30550.0
9997     999700     30550.0
9998     999800     30552.0
9999     999900     30550.0

[10000 rows x 2 columns]
      timestamp  mean_price
0             0     30559.0
1           100     30559.0
2           200     30560.0
3           300     30562.0
4           400     30566.0
...         ...         ...
9995     999500     30552.0
9996     999600     30550.0
9997     999700     30550.0
9998     999800     30552.0
9999     999900     30550.0

[10000 rows x 2 columns]


C:\Users\MY PC\AppData\Local\Temp\ipykernel_22156\4144641485.py:15: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [65]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Example function implementing pairs trading


def pairs_trading_basket(summed_mean_prices_df, basket1_df, window=1000, z_threshold=2.0):
    """
    Summarize:
      1. Merges synthetic (summed_mean_prices_df) & actual (basket1_df)
      2. Computes rolling z-score of the spread
      3. Generates simple trading signals
      4. Plots the results
    """
    # Merge
    df_pairs = pd.merge(
        summed_mean_prices_df,
        basket1_df,
        on='timestamp',
        how='inner',
        suffixes=('_synthetic', '_actual')
    )

    # Spread
    df_pairs['spread'] = df_pairs['mean_price_actual'] - \
        df_pairs['mean_price_synthetic']

    # Rolling stats
    df_pairs['spread_mean'] = df_pairs['spread'].rolling(window).mean()
    df_pairs['spread_std'] = df_pairs['spread'].rolling(window).std()

    # Z-score
    df_pairs['zscore'] = (df_pairs['spread'] -
                          df_pairs['spread_mean']) / df_pairs['spread_std']

    # Threshold-based signals
    df_pairs['signal'] = 0
    df_pairs.loc[df_pairs['zscore'] > z_threshold, 'signal'] = -1
    df_pairs.loc[df_pairs['zscore'] < -z_threshold, 'signal'] = 1

    # --- Plotting ---
    fig = go.Figure()

    # Synthetic Basket
    fig.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs['mean_price_synthetic'],
        mode='lines',
        name='Synthetic Basket'
    ))

    # Actual Basket
    fig.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs['mean_price_actual'],
        mode='lines',
        name='Actual PICNIC_BASKET2'
    ))

    # Identify buy/sell signals
    buy_signals = df_pairs[df_pairs['signal'] == 1]
    sell_signals = df_pairs[df_pairs['signal'] == -1]
    # buy_signals = df_pairs[df_pairs['signal'] == 1]
    # sell_signals = df_pairs[df_pairs['signal'] == -1]

    fig.add_trace(go.Scatter(
        x=buy_signals['timestamp'],
        y=buy_signals['mean_price_actual'],
        mode='markers',
        marker_symbol='triangle-up',
        marker_color='green',
        marker_size=12,
        name='Buy (Long Actual)'
    ))

    fig.add_trace(go.Scatter(
        x=sell_signals['timestamp'],
        y=sell_signals['mean_price_actual'],
        mode='markers',
        marker_symbol='triangle-down',
        marker_color='red',
        marker_size=12,
        name='Sell (Short Actual)'
    ))

    fig.update_layout(
        title='Pairs Trading Signals: Actual vs Synthetic',
        xaxis_title='Timestamp',
        yaxis_title='Price'
    )

    fig.show()

    return df_pairs


# Usage:
df_signals = pairs_trading_basket(
    summed_mean_prices_df,
    price_dict["PICNIC_BASKET2"],
    window=1000,
    z_threshold=1.5
)

In [73]:
def pairs_trading_two_products(prod1, prod2, price_dict, window=20, z_threshold=2.0):
    """
    Implements pairs trading for two products by:
      1. Normalizing their price series
      2. Merging them on the timestamp
      3. Computing the spread (difference between normalized prices)
      4. Calculating the rolling mean and standard deviation of the spread
      5. Generating trading signals based on the spread's z-score
      6. Plotting the normalized prices with long/short signals and a spread z-score subplot

    Parameters:
      - prod1: str, key for the first product in price_dict
      - prod2: str, key for the second product in price_dict
      - price_dict: dict, where each key corresponds to a DataFrame containing at least 
                    a 'timestamp' column and a 'mean_price' column.
      - window: int, the lookback window (in number of data points) for the rolling calculations.
      - z_threshold: float, the threshold z-score used to trigger trading signals.

    Returns:
      - df_pairs: DataFrame containing merged data, computed spread, z-score, and signals.
    """

    # --- 1. Get the DataFrames for the two products ---
    df1 = price_dict[prod1].copy()
    df2 = price_dict[prod2].copy()

    # Ensure the timestamp column is in datetime format
    df1['timestamp'] = pd.to_datetime(df1['timestamp'])
    df2['timestamp'] = pd.to_datetime(df2['timestamp'])

    # --- 2. Normalize the Price Series ---
    # Normalization: subtract the series mean and divide by its standard deviation.
    df1['norm_price'] = (df1['mean_price'] -
                         df1['mean_price'].mean()) / df1['mean_price'].std()
    df2['norm_price'] = (df2['mean_price'] -
                         df2['mean_price'].mean()) / df2['mean_price'].std()

    # --- 3. Merge on the Timestamp ---
    # We only need the timestamp and normalized price.
    df_pairs = pd.merge(
        df1[['timestamp', 'norm_price']],
        df2[['timestamp', 'norm_price']],
        on='timestamp',
        how='inner',
        suffixes=('_' + prod1, '_' + prod2)
    )

    # --- 4. Compute the Spread and Rolling Statistics ---
    # The spread is the difference between the normalized prices.
    df_pairs['spread'] = df_pairs[f'norm_price_{prod1}'] - \
        df_pairs[f'norm_price_{prod2}']

    # Calculate the rolling mean and standard deviation of the spread.
    df_pairs['spread_mean'] = df_pairs['spread'].rolling(window).mean()
    df_pairs['spread_std'] = df_pairs['spread'].rolling(window).std()

    # Compute the z-score for the spread (you may need to ignore the initial NaNs).
    df_pairs['zscore'] = (df_pairs['spread'] -
                          df_pairs['spread_mean']) / df_pairs['spread_std']

    # --- 5. Generate Trading Signals ---
    # Here a simple rule is used:
    #   * If zscore > +z_threshold: the spread is 'too high'
    #       => Signal to SHORT product1 (and LONG product2), signal = -1
    #   * If zscore < -z_threshold: the spread is 'too low'
    #       => Signal to LONG product1 (and SHORT product2), signal = +1
    #   * Else, no signal = 0.
    df_pairs['signal'] = 0
    df_pairs.loc[df_pairs['zscore'] > z_threshold, 'signal'] = -1
    df_pairs.loc[df_pairs['zscore'] < -z_threshold, 'signal'] = 1

    # --- 6. Plotting ---
    # Create two plots:
    #   1. Normalized price series of both products with trading signals marked.
    #   2. A subplot of the spread z-score with horizontal threshold lines.

    # Plot #1: Normalized Prices and Signals
    fig = go.Figure()

    # Plot normalized prices for product 1
    fig.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs[f'norm_price_{prod1}'],
        mode='lines',
        name=f'Normalized Price {prod1}'
    ))

    # Plot normalized prices for product 2
    fig.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs[f'norm_price_{prod2}'],
        mode='lines',
        name=f'Normalized Price {prod2}'
    ))

    # Identify the rows where there are buy signals (signal = 1) and sell signals (signal = -1)
    buy_signals = df_pairs[df_pairs['signal'] == 1]
    sell_signals = df_pairs[df_pairs['signal'] == -1]

    # On the price chart, we mark the signals on product1's price (or either one)
    fig.add_trace(go.Scatter(
        x=buy_signals['timestamp'],
        y=buy_signals[f'norm_price_{prod1}'],
        mode='markers',
        marker=dict(symbol='triangle-up', color='green', size=12),
        name='Buy Signal (Long ' + prod1 + ')'
    ))
    fig.add_trace(go.Scatter(
        x=sell_signals['timestamp'],
        y=sell_signals[f'norm_price_{prod1}'],
        mode='markers',
        marker=dict(symbol='triangle-down', color='red', size=12),
        name='Sell Signal (Short ' + prod1 + ')'
    ))

    fig.update_layout(
        title=f'Normalized Prices: {prod1} vs {prod2} with Trading Signals',
        xaxis_title='Timestamp',
        yaxis_title='Normalized Price'
    )

    # Plot #2: Use a subplot with two rows.
    # The first row shows the normalized price series (and trading signals).
    # The second row shows the spread’s z-score (with threshold lines).
    fig2 = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=(
            f"Normalized Prices: {prod1} vs {prod2}", "Spread Z-score")
    )

    # Row 1: Normalized Price series
    fig2.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs[f'norm_price_{prod1}'],
        mode='lines',
        name=f'{prod1} Norm Price'
    ), row=1, col=1)

    fig2.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs[f'norm_price_{prod2}'],
        mode='lines',
        name=f'{prod2} Norm Price'
    ), row=1, col=1)

    fig2.add_trace(go.Scatter(
        x=buy_signals['timestamp'],
        y=buy_signals[f'norm_price_{prod1}'],
        mode='markers',
        marker=dict(symbol='triangle-up', color='green', size=12),
        name='Buy Signal'
    ), row=1, col=1)

    fig2.add_trace(go.Scatter(
        x=sell_signals['timestamp'],
        y=sell_signals[f'norm_price_{prod1}'],
        mode='markers',
        marker=dict(symbol='triangle-down', color='red', size=12),
        name='Sell Signal'
    ), row=1, col=1)

    # Row 2: Spread Z-score and threshold lines
    fig2.add_trace(go.Scatter(
        x=df_pairs['timestamp'],
        y=df_pairs['zscore'],
        mode='lines',
        name='Spread Z-score'
    ), row=2, col=1)

    # Add horizontal dashed lines to denote the upper and lower thresholds.
    fig2.add_hline(y=z_threshold, line_dash="dash", line_color="red",
                   annotation_text=f"+{z_threshold}", row=2, col=1)
    fig2.add_hline(y=-z_threshold, line_dash="dash", line_color="green",
                   annotation_text=f"-{z_threshold}", row=2, col=1)

    fig2.update_layout(
        height=700,
        title_text=f"Pairs Trading: {prod1} vs {prod2} (Normalized Prices & Spread Z-score)",
        xaxis2_title="Timestamp"
    )

    # Show the plots
    fig.show()
    fig2.show()

    return df_pairs


# --- Example Usage ---


In [85]:
# Define your product identifiers (they should match the keys in price_dict)
prod1 = "CROISSANTS"  # Example key for first product
prod2 = "DJEMBES"  # Example key for second product
# "CROISSANTS", "JAMS"
# Assuming price_dict is already defined and contains DataFrames for both products,
# each with at least 'timestamp' and 'mean_price' columns.
df_signals = pairs_trading_two_products(
    prod1, prod2, price_dict, window=1000, z_threshold=2)

In [10]:
def generate_for_day_normalised(day:str, products):
    # Read the data for the specified day
    file_path = f"../../prosperity3bt/resources/round2/prices_round_2_day_{day}.csv"
    data = pd.read_csv(file_path, sep=';')

    # Calculate the mean price
    data['mean_price'] = (
        data[['bid_price_1', 'bid_price_2', 'bid_price_3']].min(axis=1, skipna=True) +
        data[['ask_price_1', 'ask_price_2', 'ask_price_3']].max(axis=1, skipna=True)
    ) / 2

    # Normalize the mean_price for each product
    normalized_data = data[data['product'].isin(products)].copy()
    normalized_data['normalized_mean_price'] = normalized_data.groupby('product')['mean_price'].transform(
        lambda x: (x - x.mean()) / x.std()
    )

    # Plot the normalized data
    fig = px.line(
        normalized_data,
        x="timestamp",
        y="normalized_mean_price",
        color="product",
        title="Normalized Mean Price Over Time for Selected Products"
    )
    fig.show()

In [11]:
to_calc_products = [
    "PICNIC_BASKET1",
    "PICNIC_BASKET2",
    "CROISSANTS",
    "DJEMBES",
    "JAMS"
    ]
generate_for_day_normalised("1", to_calc_products)

In [12]:
def calculate_price_correlation(data, product1, product2):
    """
    Calculate the correlation of mean prices between two products.

    Parameters:
        data (pd.DataFrame): The dataset containing product data.
        product1 (str): The name of the first product.
        product2 (str): The name of the second product.

    Returns:
        float: The correlation coefficient between the mean prices of the two products.
    """
    # Filter data for the two products
    product1_data = data[data['product'] == product1][['timestamp', 'mean_price']].rename(columns={'mean_price': f'{product1}_mean_price'})
    product2_data = data[data['product'] == product2][['timestamp', 'mean_price']].rename(columns={'mean_price': f'{product2}_mean_price'})

    # Merge the data on timestamp
    merged_data = pd.merge(product1_data, product2_data, on='timestamp', how='inner')

    # Calculate the correlation
    correlation = merged_data[f'{product1}_mean_price'].corr(merged_data[f'{product2}_mean_price'])

    return correlation

In [13]:
cor= {}
for prod1 in to_calc_products:
    for prod2 in to_calc_products:
        if prod1 != prod2 and f"\"{prod2}\",\"{prod1}\"" not in cor:
            correlation = calculate_price_correlation(data, prod1, prod2)
            cor[f"\"{prod1}\",\"{prod2}\""]=float(correlation)

# calculate_price_correlation(data, "CROISSANTS", "PICNIC_BASKET1")

In [16]:
sorted_cor = dict(sorted(cor.items(), key=lambda item: item[1], reverse=True))
for key, value in sorted_cor.items():
    print(f"{key}: {value:.2f}")
    # generate_for_day_normalised("1", [key.split(",")[0].replace("\"", ""), key.split(",")[1].replace("\"", "")])

"PICNIC_BASKET1","PICNIC_BASKET2": 0.83
"PICNIC_BASKET1","JAMS": 0.79
"PICNIC_BASKET2","JAMS": 0.77
"DJEMBES","JAMS": 0.59
"PICNIC_BASKET1","CROISSANTS": 0.58
"CROISSANTS","DJEMBES": 0.53
"PICNIC_BASKET1","DJEMBES": 0.52
"PICNIC_BASKET2","CROISSANTS": 0.40
"PICNIC_BASKET2","DJEMBES": 0.29
"CROISSANTS","JAMS": 0.25


In [ ]:
PICNIC1 by PICNIC2 2.5

In [ ]:
from scipy.signal import correlate
import numpy as np

def calculate_lead_lag(data, product1, product2, max_lag=10):
    """
    Calculate the lead-lag relationship between two products using cross-correlation.

    Parameters:
        data (pd.DataFrame): The dataset containing product data.
        product1 (str): The name of the first product.
        product2 (str): The name of the second product.
        max_lag (int): The maximum lag to consider.

    Returns:
        dict: A dictionary with lags as keys and cross-correlation values as values.
    """
    # Filter data for the two products
    product1_data = data[data['product'] == product1].set_index('timestamp')['mean_price']
    product2_data = data[data['product'] == product2].set_index('timestamp')['mean_price']

    # Align the two time series
    aligned_data = pd.concat([product1_data, product2_data], axis=1, keys=[product1, product2]).dropna()

    # Calculate cross-correlation
    xcorr = correlate(aligned_data[product1] - aligned_data[product1].mean(),
                       aligned_data[product2] - aligned_data[product2].mean(), mode='full')
    lags = np.arange(-len(aligned_data) + 1, len(aligned_data))

    # Extract cross-correlation for the specified lag range
    lag_range = (lags >= -max_lag) & (lags <= max_lag)
    cross_corr = dict(zip(lags[lag_range], xcorr[lag_range]))

    return cross_corr


# Example usage
lead_lag = calculate_lead_lag(data, "CROISSANTS", "JAMS", max_lag=10)
for lag, value in lead_lag.items():
    print(f"Lag {lag}: {value:.2f}")

Lag -10: 454778.02
Lag -9: 454525.72
Lag -8: 454281.02
Lag -7: 454019.77
Lag -6: 453788.87
Lag -5: 453564.77
Lag -4: 453296.86
Lag -3: 453041.76
Lag -2: 452796.35
Lag -1: 452557.49
Lag 0: 452315.69
Lag 1: 452299.19
Lag 2: 452274.93
Lag 3: 452269.93
Lag 4: 452239.92
Lag 5: 452173.53
Lag 6: 452108.63
Lag 7: 452055.99
Lag 8: 451995.35
Lag 9: 451926.96
Lag 10: 451851.57


ValueError: Length of values (10000) does not match length of index (80000)

In [27]:
# Apply a rolling window normalization for the mean price
window_size = 1000  # Define the rolling window size
data['rolling_mean'] = data.groupby('product')['mean_price'].transform(lambda x: x.rolling(window_size, min_periods=1).mean())
data['rolling_std'] = data.groupby('product')['mean_price'].transform(lambda x: x.rolling(window_size, min_periods=1).std())

# Calculate the rolling normalized mean price
data['rolling_normalized_mean_price'] = (data['mean_price'] - data['rolling_mean']) / data['rolling_std']

In [28]:
ink= data[data["product"]=="SQUID_INK"]

In [41]:
# Plot the normalized data
fig = px.line(
    ink,
    x="timestamp",
    y='mean_price',
    title="Rolling Normalized Mean Price and Rolling Mean Over Time for SQUID_INK"
)

fig.show()

In [32]:
# Plot the normalized data
fig = px.line(
    ink,
    x="timestamp",
    y=['rolling_normalized_mean_price', 'rolling_mean'],
    title="Rolling Normalized Mean Price and Rolling Mean Over Time for SQUID_INK"
)

fig.show()

In [55]:
# Apply a rolling window normalization for the mean price
window_size = 1000  # Define the rolling window size
data['rolling_mean'] = data.groupby('product')['mean_price'].transform(
    lambda x: x.rolling(window_size, min_periods=1).mean()
)
data['rolling_std'] = data.groupby('product')['mean_price'].transform(
    lambda x: x.rolling(window_size, min_periods=1).std()
)

# Calculate the rolling normalized mean price
data['rolling_normalized_mean_price'] = (
    data['mean_price'] - data['rolling_mean']) / data['rolling_std']

# Filter the data for the specific product
ink = data[data["product"] == "SQUID_INK"]
# -------------------------------------------------------------------------------

# Define a threshold for signal generation
threshold = 2.8

# Define signals:
#   - Long if the normalized price is less than -threshold (price is unusually low)
#   - Short if the normalized price is greater than +threshold (price is unusually high)
ink.loc[:, 'position'] = 0  # Initialize a column for positions

ink.loc[(ink['rolling_normalized_mean_price'] < -threshold) & (ink['mean_price'] <= 2000), 'position'] = 1  # Long signal
ink.loc[(ink['rolling_normalized_mean_price'] > threshold) & (ink['mean_price'] >= 2000), 'position'] = -1  # Short signal

# Optionally, you might want to filter signals so that consecutive signals aren’t all repeated;
# For simplicity, we will plot every point where the condition holds.

# Create subsets for long and short signals
long_signals = ink[ink['position'] == 1]
short_signals = ink[ink['position'] == -1]

# Plot the price along with the long/short markers
fig = go.Figure()

# Plot the mean price as a line chart
fig.add_trace(go.Scatter(
    x=ink['timestamp'],
    y=ink['mean_price'],
    mode='lines',
    name='Mean Price',
    line=dict(color='blue')
))

# # Add markers for long positions (triangle-up markers)
fig.add_trace(go.Scatter(
    x=long_signals['timestamp'],
    y=long_signals['mean_price'],
    mode='markers',
    name='Long Signal',
    marker=dict(symbol='triangle-up', size=10, color='green')
))

# Add markers for short positions (triangle-down markers)
fig.add_trace(go.Scatter(
    x=short_signals['timestamp'],
    y=short_signals['mean_price'],
    mode='markers',
    name='Short Signal',
    marker=dict(symbol='triangle-down', size=10, color='red')
))
fig.add_trace(go.Scatter(
    x=ink['timestamp'],
    y=ink['rolling_normalized_mean_price'],
    mode='lines',
    name='Normalized Price',
    line=dict(color='orange'),
    yaxis='y2'  # Specify the secondary y-axis
))

# Update layout to include a secondary y-axis
fig.update_layout(
    yaxis2=dict(
        title="Normalized Price",
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white'
)
# Add horizontal lines for the threshold levels scaled to the y2 axis
fig.add_hline(y=threshold, line_dash="dash", line_color="green", annotation_text="+Threshold", annotation_position="top right", yref="y2")
fig.add_hline(y=-threshold, line_dash="dash", line_color="red", annotation_text="-Threshold", annotation_position="bottom right", yref="y2")
# fig.update_layout(
#     title='Price with Long/Short Positions for SQUID_INK',
#     xaxis_title='Timestamp',
#     yaxis_title='Mean Price',
#     template='plotly_white'
# )

fig.show()

C:\Users\MY PC\AppData\Local\Temp\ipykernel_22488\1320750276.py:24: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

